# Content Managers

In [2]:
with open('notes.txt','w') as file:
    file.write('some todoo...')
    
with open('notes.txt','r') as file:
    print(file.read())

some todoo...


In [3]:
from threading import Lock

lock = Lock()

with lock:
    shared_resource = 'Some value'

# Class based Context Manager for managing file

In [2]:
class ManagedFile:
  def __init__(self, filename, mode):
    self.filename = filename
    self.mode = mode

  def __enter__(self):
    print('  enter')
    self.file = open(self.filename, self.mode)
    return self.file

  def __exit__(self, exc_type, exc_value, exc_traceback):
    print('  exit')
    if self.file:
      self.file.close()

filename = 'greetings.txt'

print (f'Write message to file {filename}..')
with ManagedFile(filename, 'w') as file:
  file.write("Hello world!")

print()
print ('--'*20)
print()

print (f'Read message from file {filename}..')
with ManagedFile(filename, 'r') as file:
  print(file.read())


Write message to file greetings.txt..
  enter
  exit

----------------------------------------

Read message from file greetings.txt..
  enter
Hello world!
  exit


# Class based Context Manager with Exception Handling

In [4]:
class ManagedFile:
    def __init__(self, filename):
        print('init')
        self.filename = filename
        
    def __enter__(self):
        print('enter')
        self.file = open(self.filename, 'w')
        return self.file
    
    def __exit__(self, exc_type, exc_value, exc_traceback):
        if self.file:
            self.file.close()
        if exc_type is not None:
            print('exception has been handled')
        #print('Exception: ',exc_type, exc_value)
        print('exit')
        return True
        

with ManagedFile('notes.txt') as file:
    print('do some stuff...')
    file.write('some todooo...')
    file.somemethod()
    
print('continuing..')

init
enter
do some stuff...
exception has been handled
exit
continuing..


# Function based Context Manager

In [7]:
from contextlib import contextmanager

@contextmanager
def open_managed_file(filename, mode):
    file = open(filename, mode)
    try:
        yield file
    finally:
        file.close()
        
with open_managed_file('notes.txt','w') as f:
    f.write('some todoo...')

with open_managed_file('notes.txt','r') as f:
    print(f.read())

some todoo...


# Real life example with Database Connection

```
Phase 1 - Build a SQLite connection class
Phase 2 - Create a class-based context manager
```

### Phase 1 - Define `DatabaseConnection` class

In [9]:
import sqlite3

class DatabaseConnection:
    def __init__(self, db_path):
        self.db_path = db_path
        self.connection = None

    def open(self):
        self.connection = sqlite3.connect(self.db_path)

    def close(self):
        if self.connection is not None:
            self.connection.close()
            self.connection = None
    
    def create_table(self):
        query = """
            CREATE TABLE IF NOT EXISTS employees (
                id INTEGER PRIMARY KEY,
                name TEXT NOT NULL
            )
        """
    
        cursor = self.connection.cursor()
        cursor.execute(query)
        self.connection.commit()
        cursor.close()
    
    def insert_employee(self, name):
        query = "INSERT INTO employees (name) VALUES (?)"
    
        cursor = self.connection.cursor()
        cursor.execute(query, (name,))
        self.connection.commit()
        cursor.close()
    
    def get_employees(self):
        query = "SELECT id, name FROM employees"
    
        cursor = self.connection.cursor()
        cursor.execute(query)
    
        employees = cursor.fetchall()
    
        cursor.close()
        return employees


if __name__ == "__main__":
    db = DatabaseConnection("practice.db")

    try:
        db.open()
        db.create_table()

        db.insert_employee("Ajay")
        db.insert_employee("Saanvi")

        for employee in db.get_employees():
            print(employee)
    finally:
        db.close()


(1, 'Ajay')
(2, 'Saanvi')
(3, 'Ajay')
(4, 'Saanvi')


### Phase 2 - Define class based Context Manager

In [13]:
class ManagedConnection:
    def __init__(self, db_path):
        self.db_path = db_path
        self.dbObj = None

    def __enter__(self):
        dbObj = DatabaseConnection(self.db_path)
        dbObj.open()
        self.dbObj = dbObj
        return self.dbObj

    def __exit__(self, exc_type, exc_value, exc_traceback):
        if self.dbObj is not None:
            self.dbObj.close()

if __name__ == "__main__":
    with ManagedConnection("practice.db") as db:
        db.insert_employee("Anant")
        db.insert_employee("Priti")

        for employee in db.get_employees():
            print(employee)
        

(1, 'Ajay')
(2, 'Saanvi')
(3, 'Ajay')
(4, 'Saanvi')
(5, 'Anant')
(6, 'Priti')
